# [개념 정리]

## Ch7. 앙상블 학습과 랜덤 포레스트
* 앙상블 학습: 일련의 예측기(앙상블)로부터 예측을 수집하면 가장 좋은 모델 하나보다 더 좋은 예측을 얻을 수 있는것
* 랜덤포레스트: 예측을 하려면 모든 개별 트리의 예측을 구하고 그런다음 가장 많은 선택을 받은 클래스를 예측을 삼는다. 이 결정트리의 앙상블을 말한다.

### 7.1 투표 기반 분류기
* 직접투표 : 각 분류기의 예측을 모아서 가장 많이 선택된 클래스를 예측하는 것으로 다수결 투표로 정해지는 분류기를 말함
* 각 분류기가 약한 학습기일지라도 충분하게 많고 다양하면 앙상블은 강한 학습기가 될 수 있음

--> why? **큰수의 법칙**

* 간접 투표: 모든 분류기가 클래스의 확률을 예측할 수 있으면, 개별 분류기의 예측을 평균 내어 확률이 가장 높은 클래스를 예측하는 것
* voting='soft'로 하여



### 7.2 배깅과 페이스팅
* 배깅: 훈련 세트에서 중복을 허용하여 샘플링하는 방식
* 페이스팅: 중복을 허용하지 않고 샘플링하는 방식

-> 공통점: 같은 훈련샘플을 여러개의 예측기에 걸쳐 사용 가능

-> 차이점: 배깅만이 한 예측기를 위해 같은 훈련샘플을 여러번 샘플링할 수 있음

  ### 7.2.1 사이킷런의 배깅과 페이스팅
  *Bagging Classifier을 이용함.
  * if 패이스팅을 이용 -> bootstrap=False로 지정하면 됨.
  * 편향 : 배깅 >페이스팅

  ### 7.2.2 oob평가
  *배깅 사용시 : 어떤 샘플은 한 예측기를 위해 여러번 샘플링되고 어떤 것은 전혀 선택되지 않을 수 있음

  -> 이때, 선택되지 않은 샘플 : oob 샘플

### 7.3 랜덤 패치와 랜덤 서브스페이스
* 샘플링 매개변수 : max_features, bootstrap_feature <- 특성에 대한 샘플링 임
* 랜덤 패치 방식 : 훈련과 샘플을 모두 샘플링하는 것
* 랜덤서브스페이스 : 훈련 샘플을 모두 사용하고 특성은 샘플링하는 것

### 7.4 랜덤 포레스트
* 배깅 방법을 적용한 결정 트리의 앙상블임
* 알고리즘 : 무작위로 선택한 특성 후보 중에서 최적의 특성을 찾는 식으로 무작위성을 더 주입함.
* max_sample : 훈련 세트의 크기 지정

 ### 7.4.1 엑스트라 트리
 * 액스트라 랜덤 트리 : 극단적으로 무작위한 트리의 랜덤 포레스트를 말함 (ExtraTreeClassifier를 사용함)

 ### 7.4.2 특성 중요도
 * feature_importance_변수 이용
 * 장점: 특성의 상대적 중요도를 측정하기 쉽다는 것
 * 특성의 중요도 측정 방법: 평균적으로 불순도를 얼마나 감소시키는지 확인하여 진행

In [1]:
import warnings
warnings.filterwarnings('ignore')

# import package
import numpy as np
import os

# 5장에서의 moons dataset 불러오기
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
X,y = make_moons(n_samples=100, noise=0.15)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf=LogisticRegression()
rnd_clf=RandomForestClassifier()
svm_clf=SVC()

voting_clf=VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf),('svc', svm_clf)],
    voting='hard')
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [3]:
from sklearn.metrics import accuracy_score
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
  clf.fit(X_train, y_train)
  y_pred=clf.predict(X_test)
  print(clf.__class__,__name__, accuracy_score(y_test, y_pred))

<class 'sklearn.linear_model._logistic.LogisticRegression'> __main__ 0.9
<class 'sklearn.ensemble._forest.RandomForestClassifier'> __main__ 0.9
<class 'sklearn.svm._classes.SVC'> __main__ 1.0
<class 'sklearn.ensemble._voting.VotingClassifier'> __main__ 1.0


In [4]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf=BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, max_samples=50, bootstrap=True, n_jobs=-1)
bag_clf.fit(X_train, y_train)
y_pred=bag_clf.predict(X_test)

In [5]:
### oob 평가
bag_clf=BaggingClassifier(DecisionTreeClassifier(), n_estimators=500, bootstrap=True, n_jobs=-1, oob_score=True)
bag_clf.fit(X_train, y_train)
bag_clf.oob_score_

0.9375

In [6]:
from sklearn.metrics import accuracy_score
y_pred=bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.95

In [7]:
bag_clf.oob_decision_function_

array([[0.07514451, 0.92485549],
       [0.00502513, 0.99497487],
       [0.03      , 0.97      ],
       [0.02173913, 0.97826087],
       [1.        , 0.        ],
       [0.9950495 , 0.0049505 ],
       [0.97058824, 0.02941176],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.97927461, 0.02072539],
       [0.        , 1.        ],
       [0.78977273, 0.21022727],
       [1.        , 0.        ],
       [0.92063492, 0.07936508],
       [1.        , 0.        ],
       [0.88829787, 0.11170213],
       [0.01522843, 0.98477157],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.98404255, 0.01595745],
       [0.67777778, 0.32222222],
       [0.02702703, 0.97297297],
       [1.        , 0.        ],
       [0.        , 1.        ],
       [0.07027027, 0.92972973],
       [0.        , 1.        ],
       [0.0212766 , 0.9787234 ],
       [0.

In [8]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf=RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
rnd_clf.fit(X_train, y_train)

y_pred_rf=rnd_clf.predict(X_test)

In [9]:
bag_clf=BaggingClassifier(DecisionTreeClassifier(max_features='auto', max_leaf_nodes=16), n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1)

In [10]:
from sklearn.datasets import load_iris
iris=load_iris()
rnd_clf=RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(iris['data'], iris['target'])
for name, score in zip(iris['feature_names'], rnd_clf.feature_importances_):
  print(name, score)

sepal length (cm) 0.1036851542593565
sepal width (cm) 0.02406867577357732
petal length (cm) 0.4210262194630058
petal width (cm) 0.4512199505040604
